In [0]:

from pyspark.sql import functions as F
from pyspark.ml.feature import Bucketizer
import math


# ============================================================
# CONFIG
# ============================================================

REFERENCE_TABLE = "fraud_mlops.gold.fraud_training_snapshot"

CURRENT_SOURCE = (
    "fraud_mlops.gold.production_drift_reference"
)

OUTPUT_TABLE = "fraud_mlops.gold.data_drift_metrics"

PSI_THRESHOLD = 0.20
EPSILON = 1e-6


FEATURES = [
    "amount",
    "log_amount",
    "type_cash_in",
    "type_cash_out",
    "type_debit",
    "type_payment",
    "type_transfer",
    "origin_txn_count_prior",
    "origin_avg_amount_prior",
    "origin_total_amount_prior",
    "amount_vs_origin_avg",
    "origin_has_history",
]


BINARY_FEATURES = {
    "type_cash_in",
    "type_cash_out",
    "type_debit",
    "type_payment",
    "type_transfer",
    "origin_has_history",
}


# ============================================================
# LOAD DATA
# ============================================================

reference_df = (
    spark.table(REFERENCE_TABLE)
    .filter(F.col("dataset_split") == "train")
    .select(*FEATURES)
)

current_df = spark.table(CURRENT_SOURCE)


reference_total = reference_df.count()
current_total = current_df.count()

print("Training reference rows:", reference_total)
print("Current production rows:", current_total)


if reference_total == 0 or current_total == 0:
    raise ValueError("Reference or current dataset is empty.")


# ============================================================
# COMMON PSI CALCULATION
# ============================================================

def calculate_psi_from_counts(
    reference_counts,
    current_counts,
    join_column
):

    distribution = (
        reference_counts
        .join(
            current_counts,
            on=join_column,
            how="full"
        )
        .fillna(0)
        .withColumn(
            "reference_pct",
            F.greatest(
                F.col("reference_count") / reference_total,
                F.lit(EPSILON)
            )
        )
        .withColumn(
            "current_pct",
            F.greatest(
                F.col("current_count") / current_total,
                F.lit(EPSILON)
            )
        )
        .withColumn(
            "psi",
            (
                F.col("current_pct")
                - F.col("reference_pct")
            )
            *
            F.log(
                F.col("current_pct")
                / F.col("reference_pct")
            )
        )
    )

    return float(
        distribution
        .agg(F.sum("psi").alias("psi"))
        .first()["psi"]
        or 0.0
    )


# ============================================================
# BINARY FEATURE PSI
# ============================================================

def binary_psi(feature):

    reference_counts = (
        reference_df
        .groupBy(feature)
        .count()
        .withColumnRenamed("count", "reference_count")
    )

    current_counts = (
        current_df
        .groupBy(feature)
        .count()
        .withColumnRenamed("count", "current_count")
    )

    return calculate_psi_from_counts(
        reference_counts,
        current_counts,
        feature
    )


# ============================================================
# NUMERIC FEATURE PSI
# ============================================================

def numeric_psi(feature):

    # Decile boundaries learned ONLY from historical training data
    quantiles = reference_df.approxQuantile(
        feature,
        [0.1, 0.2, 0.3, 0.4, 0.5,
         0.6, 0.7, 0.8, 0.9],
        0.01
    )

    quantiles = sorted(
        set(
            float(x)
            for x in quantiles
            if x is not None and math.isfinite(x)
        )
    )

    if not quantiles:
        return 0.0

    splits = [
        float("-inf"),
        *quantiles,
        float("inf")
    ]

    bucketizer = Bucketizer(
        splits=splits,
        inputCol=feature,
        outputCol="bucket"
    )

    reference_counts = (
        bucketizer
        .transform(reference_df.select(feature))
        .groupBy("bucket")
        .count()
        .withColumnRenamed("count", "reference_count")
    )

    current_counts = (
        bucketizer
        .transform(current_df.select(feature))
        .groupBy("bucket")
        .count()
        .withColumnRenamed("count", "current_count")
    )

    return calculate_psi_from_counts(
        reference_counts,
        current_counts,
        "bucket"
    )


# ============================================================
# RUN DRIFT CHECK FOR ALL 12 FEATURES
# ============================================================

results = []

for feature in FEATURES:

    if feature in BINARY_FEATURES:
        psi = binary_psi(feature)
    else:
        psi = numeric_psi(feature)

    status = (
        "DRIFT"
        if psi >= PSI_THRESHOLD
        else "PASS"
    )

    results.append(
        (
            feature,
            float(psi),
            PSI_THRESHOLD,
            status
        )
    )


# ============================================================
# RESULTS
# ============================================================

results_df = spark.createDataFrame(
    results,
    [
        "feature",
        "drift_score",
        "threshold",
        "status"
    ]
)


drifted_features = sum(
    1
    for row in results
    if row[3] == "DRIFT"
)

overall_drift = drifted_features > 0


results_df = (
    results_df
    .withColumn(
        "overall_drift",
        F.lit(overall_drift)
    )
    .withColumn(
        "run_timestamp",
        F.current_timestamp()
    )
    .withColumn(
        "reference_source",
        F.lit(REFERENCE_TABLE)
    )
    .withColumn(
        "current_source",
        F.lit(CURRENT_SOURCE)
    )
)


display(
    results_df.orderBy(
        F.desc("drift_score")
    )
)


print("Drifted features:", drifted_features)
print("OVERALL DRIFT:", overall_drift)


# ============================================================
# SAVE MONITORING RESULT
# ============================================================

(
    results_df
    .write
    .format("delta")
    .mode("append")
    .saveAsTable(OUTPUT_TABLE)
)

print("Results saved to:", OUTPUT_TABLE)


